In [1]:
import pandas as pd
import numpy as np

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [3]:
master = pd.read_csv(
    'cleaned_master_dataset.csv'
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [4]:
master.shape

(82365, 39)

## **Check Required Columns**

In [6]:
[
    c for c in master.columns
    if 'customer' in c.lower()
]

['customer_id',
 'order_delivered_customer_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

## **Check Purchase Timestamp**

In [7]:
[
    c for c in master.columns
    if 'purchase' in c.lower()
]

['order_purchase_timestamp',
 'purchase_year',
 'purchase_month',
 'purchase_quarter',
 'purchase_day']

## **Convert Date Column**

In [13]:
master['order_purchase_timestamp'] = pd.to_datetime(
    master['order_purchase_timestamp'],
    format='mixed',
    dayfirst=True
)

print("Date Converted Successfully")

Date Converted Successfully


## **Create Purchase Month**

In [14]:
master['PurchaseMonth'] = (
    master['order_purchase_timestamp']
    .dt.to_period('M')
)

print("Purchase Month Created")

Purchase Month Created


## **Verify Purchase Month**

In [15]:
master[
    [
        'customer_unique_id',
        'order_purchase_timestamp',
        'PurchaseMonth'
    ]
].head()

,customer_unique_id,order_purchase_timestamp,PurchaseMonth
0,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:00,2017-10
1,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:00,2017-10
2,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:00,2017-10
3,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:00,2018-07
4,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:00,2018-08


# Create Customer Cohorts

In [16]:
cohort = master.groupby('customer_unique_id')['PurchaseMonth'].min()

cohort = cohort.reset_index()

cohort.columns = [
    'customer_unique_id',
    'CohortMonth'
]

cohort.head()

,customer_unique_id,CohortMonth
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05
1,0000f46a3911fa3c0805444483337064,2017-03
2,0004aac84e0df4da2b147fca70cf8255,2017-11
3,00053a61a98854899e70ed204dd4bafe,2018-02
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03


# Merge Cohort Information

In [17]:
master = pd.merge(
    master,
    cohort,
    on='customer_unique_id',
    how='left'
)

master[
    [
        'customer_unique_id',
        'PurchaseMonth',
        'CohortMonth'
    ]
].head()

,customer_unique_id,PurchaseMonth,CohortMonth
0,7c396fd4830fd04220f754e42b4e5bff,2017-10,2017-09
1,7c396fd4830fd04220f754e42b4e5bff,2017-10,2017-09
2,7c396fd4830fd04220f754e42b4e5bff,2017-10,2017-09
3,af07308b275d755c9edb36a90c618231,2018-07,2018-07
4,3a653a41f6f9fc3d2a113cf8398680e8,2018-08,2018-08


# Calculate Cohort Index

In [18]:
purchase_year = master['PurchaseMonth'].dt.year
purchase_month = master['PurchaseMonth'].dt.month

cohort_year = master['CohortMonth'].dt.year
cohort_month = master['CohortMonth'].dt.month

master['CohortIndex'] = (
    (purchase_year - cohort_year) * 12
    + (purchase_month - cohort_month)
    + 1
)

master[
    [
        'PurchaseMonth',
        'CohortMonth',
        'CohortIndex'
    ]
].head()

,PurchaseMonth,CohortMonth,CohortIndex
0,2017-10,2017-09,2
1,2017-10,2017-09,2
2,2017-10,2017-09,2
3,2018-07,2018-07,1
4,2018-08,2018-08,1


In [19]:
master['CohortIndex'].describe()

,CohortIndex
count,82365.000000
mean,1.076659
std,0.768756
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,20.000000


# Create Cohort Table

In [20]:
cohort_data = master.groupby(
    ['CohortMonth', 'CohortIndex']
)['customer_unique_id'].nunique().reset_index()

cohort_data.head()

,CohortMonth,CohortIndex,customer_unique_id
0,2016-09,1,4
1,2016-10,1,228
2,2016-10,7,1
3,2016-10,14,1
4,2016-10,20,1


# Create Retention Matrix

In [21]:
retention = cohort_data.pivot(
    index='CohortMonth',
    columns='CohortIndex',
    values='customer_unique_id'
)

retention.head()

CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,20
CohortMonth,,,,,,,,,,,,,,,,,,,
2016-09,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,228.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0
2016-12,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,539.0,NaN,1.0,1.0,2.0,NaN,1.0,1.0,1.0,NaN,NaN,NaN,2.0,1.0,NaN,NaN,1.0,3.0,NaN
2017-02,1247.0,3.0,2.0,1.0,6.0,NaN,3.0,1.0,2.0,3.0,2.0,3.0,2.0,2.0,2.0,NaN,1.0,2.0,NaN


# Calculate Retention Percentage

In [22]:
cohort_sizes = retention.iloc[:, 0]

retention_rate = retention.divide(
    cohort_sizes,
    axis=0
)

retention_rate.head()

CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,20
CohortMonth,,,,,,,,,,,,,,,,,,,
2016-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,1.0,NaN,NaN,NaN,NaN,NaN,0.004386,NaN,NaN,NaN,NaN,NaN,NaN,0.004386,NaN,NaN,NaN,NaN,0.004386
2016-12,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,1.0,NaN,0.001855,0.001855,0.003711,NaN,0.001855,0.001855,0.001855,NaN,NaN,NaN,0.003711,0.001855,NaN,NaN,0.001855,0.005566,NaN
2017-02,1.0,0.002406,0.001604,0.000802,0.004812,NaN,0.002406,0.000802,0.001604,0.002406,0.001604,0.002406,0.001604,0.001604,0.001604,NaN,0.000802,0.001604,NaN


In [23]:
retention_rate.to_csv(
    'cohort_retention.csv'
)

print("Cohort Retention Saved")

Cohort Retention Saved


In [24]:
import os
os.listdir()

['.config',
 'cohort_retention.csv',
 'cleaned_master_dataset.csv',
 'sample_data']